# A scheme, and the file that decides who benefits

A difference map is the most persuasive object a transport model produces. One
colour ramp, one caption, and the question of who gains from a scheme looks
settled.

This notebook makes two of them, for one scheme. The Tyne and Wear Metro
extension to Washington - 13 km from Pelaw to South Hylton, three new stations
proposed - is written into the cost matrix twice. Neither version is wrong. They
disagree about which origin-destination pairs a Metro extension makes cheaper,
and that disagreement is the whole of the difference between the two maps.

Beta stays at 0.1185 per minute across both runs and the opportunity variable
stays at jobs. What moves is a file you can open and read.

Work through the sections in order, from the top of the page to the bottom.

## Where the files are

JupyterLite runs inside your browser. Nothing is installed on your machine and
you do not need administrator rights, which is why this page opens on a
locked-down work machine.

The notebook and its data arrived with the site, so there is nothing to download
and nothing to upload. Open the file browser - the panel down the left-hand
side, or the folder icon in the far-left sidebar if it is not showing - and you
will find this arrangement already in place:

```
scheme-scenario/
    scheme-test.ipynb
    data/
        zones_msoa.csv
        trip_ends_msoa.csv
        cost_matrix_msoa.csv
        zones_msoa.geojson
        scheme_cost_adjustments_narrow.csv
        scheme_cost_adjustments_broad.csv
```

Every path in the code below assumes it. The notebook sits at the top of the
folder and the data sits one level under it, so moving either one will break the
loading section.

**What happens to anything you change**

Because there is no server behind this, whatever you save goes into your
browser's own storage rather than onto a network drive. That has two
consequences worth taking seriously. Anything you want to keep should be
downloaded - right-click the file in the file browser and choose **Download**.
And if you clear your browsing data, or if your employer's IT policy clears it
for you, your saved work goes with it.

Do not edit the CSV files. If you want to try something out on them, duplicate
one first and work on the copy.

**Getting back to the original**

Should you change the notebook and want the version you started with, use
**Help > Clear Browser Data**. But read the warning it gives you before
confirming. It removes everything you have stored on this site, for every
notebook here, and it cannot be undone, so download anything you care about
first.

## Checking the files are where you think they are

Run the cell below before anything else. It reports what it can see, which is
faster than reading an error message later and guessing what went wrong.

In [ ]:
import os

DATA_FOLDER = "data"

expected = [
    "zones_msoa.csv",
    "trip_ends_msoa.csv",
    "cost_matrix_msoa.csv",
    "zones_msoa.geojson",
    "scheme_cost_adjustments_narrow.csv",
    "scheme_cost_adjustments_broad.csv",
]

print("Looking in:", os.path.abspath(DATA_FOLDER))
print()

if not os.path.isdir(DATA_FOLDER):
    print("That folder does not exist yet.")
    print("Check the folder names and check where this notebook is saved.")
else:
    found = sorted(os.listdir(DATA_FOLDER))
    for name in expected:
        status = "found" if name in found else "MISSING"
        print(f"  {name:38s} {status}")

## Reading the scheme before running it

An intervention buried in notebook code cannot be audited. Anyone reviewing the
work has to read Python to find out what was assumed, and almost nobody does. So
the scheme here is two CSV files, and the cell below prints their comment
headers in full rather than paraphrasing them, because the header is where the
assumptions were written down.

Read both headers now. The narrow file connects the seven Washington MSOAs to
the three zones holding the regional centres. The broad file connects the same
seven origins to every zone in Tyne and Wear that already has a Metro or heavy
rail station, on the argument that an extension joins a network rather than
three destinations. Both apply the same twelve minutes to every pair they list,
so anything that differs between your two runs differs because of the pair list
and nothing else. That twelve minutes is a course assumption rather than a Nexus
figure, and the header shows the arithmetic that should make you suspicious of
it.

In [ ]:
def print_header(path):
    with open(path, encoding="utf-8-sig") as handle:
        for line in handle:
            if not line.startswith("#"):
                break
            print(line.rstrip())


import pandas as pd

for tag in ["narrow", "broad"]:
    path = f"{DATA_FOLDER}/scheme_cost_adjustments_{tag}.csv"
    print("=" * 78)
    print_header(path)
    table = pd.read_csv(path, encoding="utf-8-sig", comment="#")
    print(f"  Rows: {len(table)}"
          f"   Distinct origins: {table['origin_id'].nunique()}"
          f"   Distinct destinations: {table['destination_id'].nunique()}")
    print(f"  Change applied: {table['gc_change_minutes'].min():.2f} to "
          f"{table['gc_change_minutes'].max():.2f} minutes")
    print()

## Parameters

This is the only cell in the notebook you will change. Everything below it reads
these two values and does as it is told.

SCENARIO takes `"base"` or `"scheme"`. On `"base"` the cost matrix is left
exactly as it arrived and no scheme file is read at all.

SCHEME_DEFINITION takes `"narrow"` or `"broad"` and names which of the two files
is applied. It is ignored when SCENARIO is `"base"`.

Beta is fixed at 0.1185 per minute of generalised cost, the value calibrated at
this scale earlier in the course, and the opportunity variable is fixed at
`jobs`. Neither appears in the cell below and neither is yours to move here.
Were either free to change, you could not attribute a difference between two
maps to the scheme file.

In [ ]:
# ---------------------------------------------------------------------------
# PARAMETERS
# ---------------------------------------------------------------------------

SCENARIO = "base"              # "base" or "scheme"

SCHEME_DEFINITION = "narrow"   # "narrow" or "broad"
                               # ignored when SCENARIO is "base"

# ---------------------------------------------------------------------------

## Loading the data

Four files, of which the cost matrix is much the largest at 21,025 rows, so give
the cell a moment before deciding it has stalled. The zone list fixes the order
of everything else, so that row three of the cost matrix and row three of the
jobs column refer to the same place.

The cost used is `gc_min`, generalised cost in minutes. Beta is expressed per
minute, so the two agree.

In [ ]:
import numpy as np

BETA = 0.1185          # fixed, per minute of generalised cost
OPPORTUNITY = "jobs"   # fixed

zones = pd.read_csv(f"{DATA_FOLDER}/zones_msoa.csv", encoding="utf-8-sig")
trip_ends = pd.read_csv(f"{DATA_FOLDER}/trip_ends_msoa.csv",
                        encoding="utf-8-sig")
costs = pd.read_csv(f"{DATA_FOLDER}/cost_matrix_msoa.csv",
                    encoding="utf-8-sig")

zone_ids = list(zones["zone_id"])
position = {zone: i for i, zone in enumerate(zone_ids)}

details = (zones[["zone_id", "zone_name", "local_authority"]]
           .merge(trip_ends[["zone_id", "jobs"]], on="zone_id")
           .set_index("zone_id")
           .reindex(zone_ids))

supply = details[OPPORTUNITY].values.astype(float)

base_cost = (costs
             .pivot(index="origin_id", columns="destination_id",
                    values="gc_min")
             .reindex(index=zone_ids, columns=zone_ids)
             .values.astype(float))

print(f"Zones loaded:        {len(zone_ids)}")
print(f"Cost matrix:         {base_cost.shape[0]} by {base_cost.shape[1]}")
print(f"Opportunity column:  {OPPORTUNITY}, {supply.sum():,.0f} in total")
print(f"Beta:                {BETA} per minute")
print(f"Scenario:            {SCENARIO}", end="")
print(f"   scheme file: {SCHEME_DEFINITION}" if SCENARIO == "scheme" else "")

## Applying the scheme

Each row of the scheme file rewrites one cell of the 145 by 145 matrix. Rows the
file does not name are untouched, and the cell below says how many of each there
are.

Pause on that accounting, because it settles something the map cannot show you.
A zone's accessibility is a sum across its own row of the matrix. If no cost on
that row changes, the zone's accessibility cannot change either: not a little,
not by a rounding error, not at all. So a zone missing from the scheme file
shows exactly nothing, and a map of change is first of all a map of which pairs
somebody decided to improve.

The intrazonal diagonal is left as the accessibility work earlier in the course
left it, at half the generalised cost to the nearest other zone, and no scheme
file touches it. A zone's own opportunities enter its sum at the same weight in
both runs.

In [ ]:
scenario_cost = base_cost.copy()
rows_applied = 0

if SCENARIO == "scheme":
    adjustments = pd.read_csv(
        f"{DATA_FOLDER}/scheme_cost_adjustments_{SCHEME_DEFINITION}.csv",
        encoding="utf-8-sig", comment="#")
    for origin, destination, change in adjustments.itertuples(index=False):
        i, j = position[origin], position[destination]
        scenario_cost[i, j] = base_cost[i, j] + change
        rows_applied += 1

changed_cells = int((scenario_cost != base_cost).sum())
changed_rows = int((scenario_cost != base_cost).any(axis=1).sum())

print("WHAT THE SCHEME FILE DID TO THE MATRIX")
print("-" * 62)
print(f"Rows read from the scheme file:      {rows_applied:,}")
print(f"Cells of the matrix rewritten:       {changed_cells:,}"
      f" of {base_cost.size:,}")
print(f"Zones with at least one cost change: {changed_rows} of {len(zone_ids)}")
print(f"Zones with no change at all:         {len(zone_ids) - changed_rows}")
if rows_applied:
    touched = scenario_cost != base_cost
    print(f"Cheapest pair the file created: "
          f"{scenario_cost[touched].min():.2f} minutes of generalised cost")

## The two surfaces

One line of matrix arithmetic gives the base surface and the same line gives the
do-something surface. The table below reports the ten largest gains under
whichever file you have selected, in absolute accessibility units and as a
percentage of each zone's own starting value.

On a base run every gain is zero, which is the correct answer and a useful
check: the four figures printed at the foot of the table should match what the
accessibility notebook produced for the same beta and the same opportunity
column.

In [ ]:
base_access = np.exp(-BETA * base_cost) @ supply
scenario_access = np.exp(-BETA * scenario_cost) @ supply

change = scenario_access - base_access
percent = 100.0 * change / base_access

surface = pd.DataFrame({
    "zone_id": zone_ids,
    "zone_name": details["zone_name"].values,
    "local_authority": details["local_authority"].values,
    "base": base_access,
    "change": change,
    "percent": percent,
})

shown = surface.nlargest(10, ["change", "base"]).copy()
for column, fmt in (("base", "{:,.1f}"), ("change", "{:,.1f}"),
                    ("percent", "{:.1f}")):
    shown[column] = shown[column].map(fmt.format)

label = SCENARIO if SCENARIO == "base" else f"{SCENARIO} / {SCHEME_DEFINITION}"
print(f"TEN LARGEST GAINS      run: {label}")
print("-" * 88)
print(shown.to_string(index=False))
print()
print(f"Zones gaining anything at all: "
      f"{int((surface['change'] > 0.0001).sum())} of {len(surface)}")
print()
print("BASE SURFACE, unchanged by any scheme file")
print(f"  Highest: {surface.loc[surface['base'].idxmax(), 'zone_name']}"
      f"  {surface['base'].max():,.1f}")
print(f"  Lowest:  {surface.loc[surface['base'].idxmin(), 'zone_name']}"
      f"  {surface['base'].min():,.1f}")
print(f"  Median:  {np.median(surface['base']):,.1f}")

## Mapping the difference

Two panels, both drawn from the run you have just done. The left one is the base
surface, there so that you can see where the study area started. The right one is
the change.

Classification, stated because it decides what the map appears to show, and
because you met that argument in the mapping reading: the base panel uses
quintiles of the 145 base values; the change panel gives exactly-zero its own
class and splits the gaining zones at quartiles of the non-zero gains. Quantiles
across all 145 zones would be useless on the change panel, since most zones
change by nothing and three of the four break points would land on zero.

In [ ]:
import json
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as MplPolygon
from matplotlib.collections import PatchCollection
from matplotlib.colors import LinearSegmentedColormap

with open(f"{DATA_FOLDER}/zones_msoa.geojson", encoding="utf-8-sig") as handle:
    boundaries = json.load(handle)

rings = {}
for feature in boundaries["features"]:
    geometry = feature["geometry"]
    parts = ([geometry["coordinates"]] if geometry["type"] == "Polygon"
             else geometry["coordinates"])
    rings[feature["properties"]["zone_id"]] = [part[0] for part in parts]

GREY = "#F0EFED"
RED = "#C8102E"
GREEN = "#025944"
INK = "#1A4538"
ramp_change = LinearSegmentedColormap.from_list("ch", [GREY, RED])
ramp_base = LinearSegmentedColormap.from_list("bs", [GREY, GREEN])


def classify(values, breaks):
    return np.searchsorted(breaks, values, side="right")


base_breaks = list(np.quantile(surface["base"], [0.2, 0.4, 0.6, 0.8]))
base_class = classify(surface["base"].values, base_breaks)
base_colours = [ramp_base(0.15 + 0.85 * c / 4) for c in base_class]

gains = surface.loc[surface["change"] > 0.0001, "change"]
if len(gains) >= 4:
    change_breaks = list(np.quantile(gains, [0.25, 0.5, 0.75]))
    change_class = 1 + classify(surface["change"].values, change_breaks)
    change_class[surface["change"].values <= 0.0001] = 0
else:
    change_breaks = []
    change_class = np.zeros(len(surface), dtype=int)
change_colours = [GREY if c == 0 else ramp_change(0.25 + 0.75 * c / 4)
                  for c in change_class]

washington = set(pd.read_csv(
    f"{DATA_FOLDER}/scheme_cost_adjustments_narrow.csv",
    encoding="utf-8-sig", comment="#")["origin_id"])

fig, axes = plt.subplots(1, 2, figsize=(9.6, 6.4))
for ax, colours, heading in (
        (axes[0], base_colours, "Base accessibility"),
        (axes[1], change_colours, "Change under this run")):
    patches, face, edge, width = [], [], [], []
    for zone, colour in zip(zone_ids, colours):
        for ring in rings[zone]:
            patches.append(MplPolygon(np.array(ring), closed=True))
            face.append(colour)
            edge.append(INK if zone in washington else "white")
            width.append(1.1 if zone in washington else 0.3)
    ax.add_collection(PatchCollection(patches, facecolors=face,
                                      edgecolors=edge, linewidths=width))
    ax.autoscale_view()
    ax.set_aspect("equal")
    ax.set_axis_off()
    ax.set_title(heading, fontsize=10, color=INK)

from matplotlib.patches import Patch

def band(low, high, last=False):
    if last:
        return f"{low:,.0f} and above"
    return f"{low:,.0f} to {high:,.0f}"


left_edges = [surface["base"].min()] + base_breaks
left_labels = [band(left_edges[k], base_breaks[k] if k < 4 else 0, k == 4)
               for k in range(5)]
axes[0].legend(
    handles=[Patch(facecolor=ramp_base(0.15 + 0.85 * k / 4),
                   edgecolor="white", label=left_labels[k]) for k in range(5)],
    loc="upper left", fontsize=7, frameon=False, title="Quintiles",
    title_fontsize=7)

if change_breaks:
    right_edges = [gains.min()] + change_breaks
    right_labels = ["no change at all"] + [
        band(right_edges[k], change_breaks[k] if k < 3 else 0, k == 3)
        for k in range(4)]
    handles = [Patch(facecolor=GREY, edgecolor="white", label=right_labels[0])]
    handles += [Patch(facecolor=ramp_change(0.25 + 0.75 * (k + 1) / 4),
                      edgecolor="white", label=right_labels[k + 1])
                for k in range(4)]
    axes[1].legend(handles=handles, loc="upper left", fontsize=7,
                   frameon=False, title="Accessibility units gained",
                   title_fontsize=7)

caption = (f"Tyne and Wear, 145 MSOAs. Run: {label}. Beta {BETA} per minute, "
           f"opportunity variable {OPPORTUNITY}. Left panel classified by "
           "quintiles of base accessibility; right panel gives zones changing "
           "by nothing their own class and splits the gaining zones at "
           "quartiles of the non-zero change. The seven Washington MSOAs are "
           "outlined in both panels.")
fig.text(0.06, 0.085, caption, ha="left", va="top", fontsize=7, color=INK,
         wrap=True)
fig.subplots_adjust(left=0.02, right=0.98, top=0.95, bottom=0.10, wspace=0.02)
plt.show()

print("Break points on the change panel (accessibility units):",
      ", ".join(f"{b:,.0f}" for b in change_breaks) if change_breaks
      else "none, nothing changed")

## Where one zone's gain comes from

The map gives a zone one colour. Underneath that colour is a sum, and the sum
has a shape the map cannot show. The cell below takes whichever zone gained most
and lists the destinations that produced the gain, largest first.

Look at the top three lines and at the base cost beside them. The pattern is
not the one most people predict, and it holds across every scheme run in this
notebook.

In [ ]:
if surface["change"].max() > 0.0001:
    winner = surface.loc[surface["change"].idxmax()]
    row = position[winner["zone_id"]]

    weights_before = np.exp(-BETA * base_cost[row])
    weights_after = np.exp(-BETA * scenario_cost[row])
    contribution = (weights_after - weights_before) * supply

    parts = pd.DataFrame({
        "destination": details["zone_name"].values,
        "base_gc_min": base_cost[row],
        "jobs": supply,
        "contribution": contribution,
    })
    parts = parts[parts["contribution"] > 0.0001].nlargest(8, "contribution")
    parts["share_pct"] = 100 * parts["contribution"] / winner["change"]

    print(f"GAIN DECOMPOSED: {winner['zone_name']}")
    print(f"total change {winner['change']:,.1f} units"
          f"   ({winner['percent']:.1f} per cent of its base)")
    print("-" * 74)
    print(parts.round({"base_gc_min": 2, "jobs": 0,
                       "contribution": 1, "share_pct": 1})
          .to_string(index=False))
else:
    print("Nothing changed on this run, so there is nothing to decompose.")
    print("Set SCENARIO to \"scheme\" and run the notebook again.")

## Keeping the runs beside each other

The cell below writes this run into `scheme-run-record.csv` in this folder, keyed
on the run name, so that running the same combination twice replaces its earlier
column rather than adding a second one. Once both scheme runs exist it also
reports the zones that appear in one and not the other.

Download that file before you leave the page. It is what your written
explanation has to refer to, and comparing the two columns in Excel is quicker
than reading two maps from memory.

In [ ]:
RECORD = "scheme-run-record.csv"

this_run = surface[["zone_id", "zone_name", "local_authority", "change"]].copy()
this_run = this_run.rename(columns={"change": f"change_{label.replace(' / ', '_')}"})
this_run[this_run.columns[-1]] = this_run[this_run.columns[-1]].round(1)

if os.path.exists(RECORD):
    previous = pd.read_csv(RECORD, encoding="utf-8-sig")
    previous = previous.drop(columns=[this_run.columns[-1]], errors="ignore")
    record = previous.merge(
        this_run.drop(columns=["zone_name", "local_authority"]),
        on="zone_id", how="outer")
else:
    record = this_run

record.to_csv(RECORD, index=False, encoding="utf-8-sig")
columns = [c for c in record.columns if c.startswith("change_")]
print("RUNS RECORDED SO FAR:", ", ".join(c[7:] for c in columns))

if "change_scheme_narrow" in record.columns and \
        "change_scheme_broad" in record.columns:
    narrow_only = record[(record["change_scheme_narrow"] > 0.0001)
                         & (record["change_scheme_broad"] <= 0.0001)]
    broad_only = record[(record["change_scheme_broad"] > 0.0001)
                        & (record["change_scheme_narrow"] <= 0.0001)]
    print()
    print(f"Gaining under narrow but not broad: {len(narrow_only)}")
    print(f"Gaining under broad but not narrow: {len(broad_only)}")
    print()
    print("Largest gains that exist only under the broad file")
    print(broad_only.nlargest(6, "change_scheme_broad")
          [["zone_name", "local_authority", "change_scheme_broad"]]
          .to_string(index=False))

## What to do now

You have run the base case. Confirm the three base figures against the
accessibility work you did earlier, then do the two scheme runs.

1. Set SCENARIO to `"scheme"` and SCHEME_DEFINITION to `"narrow"`. Use **Kernel >
   Restart Kernel and Run All Cells**. Note the map, the count of zones gaining
   anything, and the decomposition.
2. Change SCHEME_DEFINITION to `"broad"` and restart and run everything again.

Write down, before you read the closing section, which of the two maps you would
have expected to see if somebody had told you only that Metro was being extended
to Washington. Then download `scheme-run-record.csv`.

## Read this once both scheme runs are finished

Four things, the last of them a matter of judgement rather than arithmetic.

The scheme is not in the model. The scheme file is. Under the narrow definition
135 of the 145 zones change by exactly nothing; under the broad definition 96 of
them do. Both runs used the same beta, the same jobs column, the same 145
polygons and the same twelve minutes, so everything separating your two maps was
settled in a text file before the model ran. The map carries no mark to say so.
That is the fourth choice of this kind you have met here, after the cumulative
threshold, the opportunity variable and the classification scheme, and it is the
one that most looks like data.

Now the result that catches people. Under the narrow file, City Centre &
Arthur's Hill gains 5,476 accessibility units, Gateshead Town 8,653 and
Sunderland Central & Deptford 6,051. None of the three gets a station and none
appears in any account of who this scheme is for. They gain because
accessibility is a property of an origin's entire opportunity set: a cheaper
link to Washington puts the 25,381 jobs in those seven MSOAs within easier reach
of Gateshead. Served by the scheme and benefiting from the scheme are different
sets.

Underneath one colour is a sum with a shape. Springwell & Usworth gains most
under the narrow file, and about two-thirds of that comes from a single
destination - City Centre & Arthur's Hill, 43,214 jobs at a base cost of 20.13
minutes. Gateshead Town sits nearer, at 16.32 minutes, and contributes roughly a
quarter as much because it holds 9,723 jobs. Sunderland is the same city as
Washington and contributes least of the three.

**Which file I would put in front of a committee.** The broad one, and not
because it is generous. Someone boarding at Washington North reaches Whitley Bay
without leaving the system, so a file denying that is asserting that the rest of
the Metro does not exist. The narrow file is the weaker representation, and it
is also the one that flatters the promoters, since it lands every modelled
benefit on the seven zones the campaign is about. But my position has a soft
spot. The broad file more than doubles the accessibility of every Washington
zone, Springwell & Usworth rising 104.7 per cent, and a measure that doubles
under one scheme is telling you that the twelve minutes is too generous or that
a flat reduction across 570 pairs is the wrong instrument. Being more nearly
right about which pairs improve has not made the run more nearly right about how
much.

**Two things this run does not do.** Land use is fixed throughout: the scheme
changes costs, accessibility responds, and not one job or resident moves in
consequence. And the study area has a single cost matrix of highway free-flow
generalised cost, so a public transport scheme has been written in as though the
road cost fell, where a real study would carry a public transport skim alongside
the highway one.